In [1]:
import argparse
import base64
import json
import os
import os.path as osp

import imgviz
import PIL.Image

from labelme.logger import logger
from labelme import utils

In [2]:
from glob import glob
import mmcv
json_files = glob('./2306/*.json')

out_dir = './2306-converted'

In [3]:
import cv2

In [4]:
import numpy as np

In [5]:
mmcv.load('/data/compressed_all_desay/coco/annotations/train.json')['categories'][0]

{'id': 0, 'name': 'free'}

In [6]:
class Category:
    def __init__(self):
        self.d = dict()
        self.counter = 1
    def __call__(self, x):
        if not  x in self.d:
            self.d[x] = self.counter
            self.counter += 1
        return self.d[x]
    
        

In [7]:
# cat2id('a')

In [8]:
cat2id = Category()
images = []
annotations = []
def f(image_id):
    json_file=json_files[image_id]
    out_dir_img = os.path.join(out_dir, 'images')
#     out_dir_annotation = os.path.join(out_dir, 'annotations', 'train.json')

    os.makedirs(out_dir, exist_ok=1)
    data = json.load(open(json_file))
    fn = os.path.basename(data['imagePath'].split('.')[0])+'.jpg'
    out_img_path = os.path.join(out_dir_img, fn)
    if not osp.exists(out_img_path):
        imageData = data.get("imageData")

        if not imageData:
            imagePath = os.path.join(os.path.dirname(json_file), data["imagePath"])
            with open(imagePath, "rb") as f:
                imageData = f.read()
                imageData = base64.b64encode(imageData).decode("utf-8")
        img = utils.img_b64_to_arr(imageData)
        mmcv.imwrite(img, out_img_path)
    images.append(dict(
        id=image_id,
        file_name=fn,
        height=data['imageHeight'], width=data['imageWidth'],
    ))
    for shape in data['shapes']:
        points = np.array(shape['points'])
        if len(points)>2:
            x,y,w,h = cv2.boundingRect(points.astype('int'))
            area = h*w
            seg = points.reshape(-1).tolist()
            annotations.append(dict(
            id=None, image_id=image_id, bbox=[x,y,w,h], segmentation=[seg], category_id=cat2id(shape['label'])
            ))
    
from avcv.process import multi_thread
multi_thread(f, range(len(json_files)));

Multi-thread Pipeline: 100%|██████████| 10191/10191 [00:44<00:00, 229.12Samples/s]

Finished


In [9]:
for i, ann in enumerate(annotations):
    ann['iscrowd'] = 0
    ann['id'] = i

cats = []
for name, id in cat2id.d.items():
    cats.append(dict(id=id, name=name))

In [10]:
out_path = os.path.join(out_dir, 'train.json')

mmcv.dump(dict(images=images, annotations=annotations, categories=cats),out_path)

In [11]:
# name_to_id

In [12]:
name_to_id = {image['file_name']:image['id'] for image in images}
id = name_to_id['stitched_image_004087.jpg']

from pycocotools.coco import COCO

coco = COCO(out_path)

[coco.anns[_] for _ in coco.getAnnIds(id)]

loading annotations into memory...
Done (t=0.21s)
creating index...
index created!


[{'id': 31,
  'image_id': 0,
  'bbox': [0, 476, 291, 299],
  'segmentation': [[0.0,
    476.15440218035315,
    290.61538461538464,
    489.70063441712927,
    265.97938144329896,
    736.6183197061264,
    0.0,
    774.5255362009716]],
  'category_id': 2,
  'iscrowd': 0},
 {'id': 32,
  'image_id': 0,
  'bbox': [0, 737, 268, 263],
  'segmentation': [[267.0103092783505,
    737.1752577319588,
    1.030927835051557,
    776.3505154639175,
    0.0,
    999.0,
    188.0,
    999.0]],
  'category_id': 2,
  'iscrowd': 0}]